# 2.4 Tellen en gemiddelden: Berichten vs. Mensen

In [02.3](02.3-simpsons_paradox.ipynb) zagen we hoe een verborgen groeperingsvariabele (zoals faculteit) een conclusie compleet kan omdraaien. 

Dit notebook laat een valkuil zien die nog fundamenteler is: **wat is de eenheid van analyse?** 
* Tel je *berichten*? 
* Of tel je *mensen*?

Die twee vragen kunnen exact het tegenovergestelde antwoord geven — en alleen de eenheid waar jouw specifieke claim over gaat, test de claim eerlijk.

In [ ]:
# Paragraaf 2.4: Benodigde tools importeren
import pandas as pd
from goad_toolkit.datatransforms import CountValues, Filter, Head, Pipeline
from goad_toolkit.visualizer import Annotate, BasePlot, PlotSettings, ScatterPlot
from scripts.pipelines import build_irc_pipeline
from scripts.plots import BarPlot
from wa_analyzer.data import load_showcase

## 2.4.1 De data: Twee IRC-chatkanalen vergelijken

We laden de vertrouwde IRC-chatdata in (`#ubuntu` en `#ubuntu-uk`), halen timestamps op en voegen de lengte van elk bericht in tekens toe (`length`):

In [ ]:
# Paragraaf 2.4.1: IRC-data inladen en berichtlengte berekenen
msgs = build_irc_pipeline().apply(load_showcase("ubuntu_irc"))
msgs["length"] = msgs.message.str.len()  # Lengte van het bericht in aantal tekens
msgs.head()

**De centrale vraag: typen mensen in het ene kanaal langere berichten dan in het andere?**

Let goed op de formulering: de vraag gaat over *mensen* (*people*). 

Kijk wat er gebeurt als we simpelweg de gemiddelde berichtlengte berekenen per bericht (`by_message`):

In [ ]:
# Paragraaf 2.4.1: Twee manieren van analyseren
# Analyse 1: Per bericht groeperen (rij = bericht)
by_message = msgs.groupby("channel").length.agg(["size", "mean", "sem"])
by_message["ci95"] = 1.96 * by_message["sem"]

# Analyse 2: Eerst per auteur aggregeren, dan pas per kanaal (rij = persoon)
per_author = msgs.groupby(["channel", "author"]).length.agg(["size", "mean"]).reset_index()
by_author = per_author.groupby("channel")["mean"].agg(["size", "mean", "sem"])
by_author["ci95"] = 1.96 * by_author["sem"]

## 2.4.2 Wat een errorbar (foutbalk) betekent en waarom hij hier hoort

Een gemiddelde berekend uit een steekproef is nooit het absolute, onfeilbare getal van de werkelijkheid. Er zit altijd meetonzekerheid in.

Een **95%-betrouwbaarheidsinterval** (*confidence interval*) geeft het bereik aan waarin het echte gemiddelde met 95% zekerheid ligt:
$$\text{ci95} = 1.96 \times \text{sem}$$
(waarbij `sem` de standaardfout van het gemiddelde is: $\text{std} / \sqrt{n}$).

* Hoe méér onafhankelijke waarnemingen ($n$) je hebt, hoe **smaller** de errorbar wordt.
* Hoe minder waarnemingen, hoe breder de foutbalk — en hoe groter de kans dat één afwijkende waarneming het hele gemiddelde vertekent.

`GroupedBarPlot` kan deze errorbars niet standaard tekenen, omdat Seaborn zijn eigen foutbalken berekent op basis van ruwe rijen, terwijl wij hier werken met reeds samengevatte gemiddelden die hun eigen intervallen meebrengen.

Daarom schrijven we een compacte `BasePlot`-subklasse (`BarPlotWithError`) die de staven tekent en vervolgens per staaf exact de juiste errorbar op de as plaatst:

In [ ]:
# Paragraaf 2.4.2: Data samenvoegen voor vergelijking en BarPlotWithError definiëren
UNITS = ["counting messages", "counting people"]

compare = pd.concat([
    by_message.assign(unit=UNITS[0]),
    by_author.assign(unit=UNITS[1]),
]).reset_index()[["channel", "unit", "mean", "ci95"]]


class BarPlotWithError(BasePlot):
    """Een gegroepeerde staafgrafiek met voorberekende betrouwbaarheidsintervallen."""

    def build(self, data: pd.DataFrame, x: str, y: str, hue: str, error: str,
              hue_order: list[str], **kwargs):
        import seaborn as sns

        # Grey-first regel: alles base_color, de laatste categorie highlight_color
        kwargs.setdefault(
            "palette",
            [self.settings.base_color] * (len(hue_order) - 1)
            + [self.settings.highlight_color],
        )
        sns.barplot(data=data, x=x, y=y, hue=hue, hue_order=hue_order,
                    ax=self.ax, **kwargs)
        if self.ax is None:
            raise ValueError("create_figure() must run before build()")

        intervals = data.set_index([x, hue])[error]
        categories = [label.get_text() for label in self.ax.get_xticklabels()]
        if len(self.ax.containers) != len(hue_order):
            raise ValueError(
                f"seaborn drew {len(self.ax.containers)} containers for "
                f"{len(hue_order)} hue levels; the error bars would be misplaced"
            )

        # Plaats de foutbalkjes exact op het midden van elke staaf:
        for container, level in zip(self.ax.containers, hue_order):
            for patch, category in zip(container, categories):
                self.ax.errorbar(
                    patch.get_x() + patch.get_width() / 2,
                    patch.get_height(),
                    yerr=intervals[(category, level)],
                    fmt="none", ecolor="black", capsize=4,
                )

        self.ax.legend(loc="lower right", framealpha=1)
        return self.fig, self.ax

### Ook deze klasse verhuist naar `scripts/plots.py`

Exact dezelfde logica als bij `BarPlot` in [02.2](02.2-comparing_categories.ipynb): code die alleen in een notebook-cel staat kan elders niet gebruikt worden. 

`BarPlotWithError` staat daarom al direct klaar in `scripts/plots.py`. In de cel hieronder importeren we hem rechtstreeks:

In [ ]:
# Paragraaf 2.4.2: Grafiek plotten met de geïmporteerde klasse
from scripts.plots import BarPlotWithError  # noqa: F811

units = PlotSettings(
    figsize=(8, 4),
    title="Same question, two units of analysis",
    xlabel="",
    ylabel="mean message length (characters)",
    base_color="#cccccc",
    highlight_color="#c44e52",
)
fig, ax = BarPlotWithError(units).plot(
    data=compare, x="channel", y="mean", hue="unit", error="ci95", hue_order=UNITS,
)

Kijk eens naar de uitkomst: **het antwoord verandert letterlijk van teken!**
* **Als je berichten telt (grijs):** `#ubuntu-uk` heeft kortere berichten dan `#ubuntu`.
* **Als je mensen telt (rood):** `#ubuntu-uk` heeft juist **langere** berichten dan `#ubuntu`!

En de errorbars zijn op beide eenheden zo smal dat dit in geen van beide gevallen op toeval berust. Beide getallen zijn wiskundig 100% juist. Maar ze beantwoorden twee wezenlijk andere vragen:
1. *Berichten tellen:* "Als ik willekeurig één los bericht pak uit dit kanaal, hoe lang is dat?"
2. *Mensen tellen:* "Als ik willekeurig één chatter pak uit dit kanaal, hoe lang zijn diens berichten gemiddeld?"

Onze claim ging over *mensen*. Slechts één van deze twee grafieken beantwoordt die vraag correct!

### Waarom keert de uitkomst om?

Het linkerpaneel hieronder toont de 8 meest actieve chatters van `#ubuntu-uk` (via een `Pipeline` met `Filter`, `CountValues` en `Head`):

In [ ]:
# Paragraaf 2.4.3: Waarom het effect omkeert (de 8 stamgasten vs de rest)
uk_channel = "#ubuntu-uk"

# Zoek de 8 drukste chatters via een pipeline:
top8 = (
    Pipeline()
    .add(Filter, expr=f"channel == '{uk_channel}'")
    .add(CountValues, column="author")
    .add(Head, n=8)
    .apply(msgs)
)
top8_share = top8.n.sum() / (msgs.channel == uk_channel).sum()

uk = per_author[per_author.channel == uk_channel]
most_verbose = uk.nlargest(1, "mean").iloc[0]

flip = PlotSettings(
    figsize=(13, 4),
    title="Why the answer flips",
    subplot_titles=[
        f"the {len(top8)} busiest authors write {top8_share:.0%} of #ubuntu-uk's messages",
        "...and the busiest are not the most verbose",
    ],
    xlabel="messages written",
    subplot_ylabels=["author", "mean message length (characters)"],
    base_color="#cccccc",
)

host = BarPlot(flip)
fig, axes = host.create_figure(n_plots=2)

# Links: De 8 drukste chatters schrijven 44% van alle berichten
host.plot_on_axes(BarPlot(flip), axes[0], data=top8, x="n", y="author",
                  color=flip.base_color)

# Rechts: De drukste chatters typen juist hele korte berichten; af en toe chatters typen lange berichten
host.plot_on_axes(ScatterPlot(flip), axes[1], data=uk, x="size", y="mean",
                  color="#4c72b0", alpha=0.6)
host.plot_on_axes(
    Annotate(flip), axes[1],
    text=f"{most_verbose.author}: {int(most_verbose['size'])} messages, "
         f"{most_verbose['mean']:.0f} chars each",
    xy=(most_verbose["size"], most_verbose["mean"]),
    xytext=(most_verbose["size"] + uk["size"].max() * 0.15,
           most_verbose["mean"] - uk["mean"].max() * 0.15),
)

Twee krachten trekken in tegengestelde richting:
1. **Het bericht-gemiddelde** wordt compleet gedomineerd door een handjevol stamgasten (de 8 drukste chatters schrijven maar liefst **44% van alle berichten** in `#ubuntu-uk`!). En zij typen extreem korte, functionele antwoorden.
2. **Het persoon-gemiddelde** geeft iedereen exact één stem: een account dat af en toe meedoet en lange alinea's typt (zoals geannoteerd in het rechterpaneel) telt bij het tellen van mensen net zo zwaar mee als een stamgast.

**De belangrijkste les om te onthouden:**
> **Hoe minder eenheden je hebt, hoe meer één extreme eenheid het antwoord kan verplaatsen.** 

Data opschonen is niet minder, maar juist véél belangrijker wanneer je aggregeert naar mensen in plaats van naar berichten.

> **Stel jezelf deze drie vragen bij elke groepsvergelijking:**
> 1. Waar gaat mijn inhoudelijke bewering werkelijk over? (Berichten, personen, dagen, gesprekken?)
> 2. Is dat ook daadwerkelijk de eenheid die ik geteld heb?
> 3. Hoeveel onafhankelijke eenheden zijn er? (Niet het aantal rijen, maar het aantal échte eenheden waarop de claim rust!)

---

**Hoe we nu verder gaan:** In [02.5-your_turn](02.5-your_turn.ipynb) ga je met je eigen WhatsApp-chat aan de slag, waarbij `BarPlot` en `BarPlotWithError` klaarliggen en jij zelf bepaalt hoe je jouw verhaal formuleert!